# Day Trading Analysis Notebook

This notebook performs a fully automated analysis of a MetaTrader-style day trading CSV export.

Expected Colab upload filename: **`day_1.csv`**

It will generate:

- clean closed-fills and aggregated closing-order event CSVs
- summary, risk, symbol, hourly, side, and trade-quality stats
- equity curve, drawdown, P&L distribution, symbol/hour charts, and more
- a self-contained HTML report
- a ZIP bundle containing all outputs


## 1. Setup

Run all cells from top to bottom. In Google Colab, upload `day_1.csv` when prompted, or place it in the working directory first.


In [ ]:
# =========================
# Configuration
# =========================

FILE_NAME = "day_1.csv"        # Colab upload name expected by default
OUTPUT_DIR = "day_trading_analysis_output"
ASSET_DIR = f"{OUTPUT_DIR}/assets"
CURRENCY_SYMBOL = ""           # Example: "$", "€", "£"; keep blank if broker export currency is implicit
INITIAL_EQUITY = None          # Optional. Example: 10000. If set, return % and DD % will be calculated.

# Behaviour
AGGREGATE_BY_CLOSING_ORDER = True  # Treat multiple fills under the same closing order as one exit event.
INCLUDE_COMMISSION_FEE_SWAP = True  # Net P&L = Profit + Commission + Fee + Swap when available.

# Chart settings
FIG_DPI = 160
FIGSIZE_WIDE = (12, 5.8)
FIGSIZE_TALL = (10, 6)
TOP_N_TABLES = 10

In [ ]:
# =========================
# Imports
# =========================

import os
import re
import math
import json
import base64
import zipfile
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, HTML, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(ASSET_DIR).mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
# =========================
# Load CSV
# =========================

def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if not os.path.exists(FILE_NAME):
    if running_in_colab():
        from google.colab import files
        print(f"{FILE_NAME} was not found in the current Colab directory.")
        print("Upload your broker CSV now. The notebook expects day_1.csv, but it can use another uploaded CSV if needed.")
        uploaded = files.upload()
        if FILE_NAME not in uploaded:
            csv_candidates = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
            if not csv_candidates:
                raise FileNotFoundError("No CSV file was uploaded.")
            FILE_NAME = csv_candidates[0]
            print(f"Using uploaded file: {FILE_NAME}")
    else:
        raise FileNotFoundError(f"Could not find {FILE_NAME}. Put the CSV next to this notebook or change FILE_NAME.")

raw = pd.read_csv(FILE_NAME, dtype=str, encoding="utf-8-sig")
raw.columns = [str(c).strip() for c in raw.columns]

print(f"Loaded {len(raw):,} rows from {FILE_NAME}")
display(raw.head())
display(pd.DataFrame({"column": raw.columns}))

In [ ]:
# =========================
# Helpers
# =========================

def find_col(df: pd.DataFrame, aliases, required=True):
    """Find a column by case-insensitive aliases."""
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for alias in aliases:
        key = alias.strip().lower()
        if key in lower_map:
            return lower_map[key]
    # fuzzy: remove spaces/underscores
    norm_map = {re.sub(r"[\s_]+", "", str(c).strip().lower()): c for c in df.columns}
    for alias in aliases:
        key = re.sub(r"[\s_]+", "", alias.strip().lower())
        if key in norm_map:
            return norm_map[key]
    if required:
        raise KeyError(f"Could not find required column. Tried aliases: {aliases}. Available columns: {list(df.columns)}")
    return None

def clean_number(x):
    """
    Convert broker-formatted numeric strings to floats.
    Handles examples like:
    - '4 155.710' -> 4155.710
    - '- 55.27' -> -55.27
    - ' 1.32162' -> 1.32162
    - '0.00' -> 0
    """
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "":
        return np.nan
    s = s.replace("\u00a0", " ")
    s = s.replace("−", "-")
    s = re.sub(r"\s+", "", s)
    # If both comma and dot exist, assume comma is thousands separator.
    if "," in s and "." in s:
        s = s.replace(",", "")
    # If only comma exists, assume it might be a decimal comma.
    elif "," in s and "." not in s:
        s = s.replace(",", ".")
    s = re.sub(r"[^0-9.\-+eE]", "", s)
    if s in {"", "-", "+", ".", "-.", "+."}:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan

def clean_id(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().replace("\u00a0", " ")
    s = re.sub(r"\s+", "", s)
    # Avoid turning IDs into floats if possible.
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    return s

def safe_div(num, den):
    if den is None or den == 0 or pd.isna(den):
        return np.nan
    return num / den

def money(x, currency=CURRENCY_SYMBOL):
    if pd.isna(x):
        return "n/a"
    return f"{currency}{x:,.2f}"

def pct(x):
    if pd.isna(x):
        return "n/a"
    return f"{x:.2%}"

def pct_points(x):
    if pd.isna(x):
        return "n/a"
    return f"{x:.1f}%"

def weighted_avg(values, weights):
    values = pd.Series(values).astype(float)
    weights = pd.Series(weights).astype(float).abs()
    mask = values.notna() & weights.notna()
    if not mask.any() or weights[mask].sum() == 0:
        return values[mask].mean() if mask.any() else np.nan
    return np.average(values[mask], weights=weights[mask])

def max_streak(sign_series, target):
    max_s = cur = 0
    for v in sign_series:
        if v == target:
            cur += 1
            max_s = max(max_s, cur)
        else:
            cur = 0
    return max_s

def signed_label(pnl):
    if pnl > 0:
        return "Win"
    if pnl < 0:
        return "Loss"
    return "Breakeven"

def side_from_closing_type(close_type):
    """
    MetaTrader convention:
    - A closing sell usually closes a long/buy position.
    - A closing buy usually closes a short/sell position.
    """
    t = str(close_type).strip().lower()
    if t == "sell":
        return "Long position closed"
    if t == "buy":
        return "Short position closed"
    return "Unknown"

In [ ]:
# =========================
# Clean and prepare deals
# =========================

COL_TIME = find_col(raw, ["Time", "Open Time", "Close Time", "Date"])
COL_DEAL = find_col(raw, ["Deal", "Deal ID", "Ticket"], required=False)
COL_SYMBOL = find_col(raw, ["Symbol", "Instrument"])
COL_TYPE = find_col(raw, ["Type", "Deal Type"])
COL_DIRECTION = find_col(raw, ["Direction", "Entry"])
COL_VOLUME = find_col(raw, ["Volume", "Lots", "Size"])
COL_PRICE = find_col(raw, ["Price"])
COL_ORDER = find_col(raw, ["Order", "Order ID"], required=False)
COL_COMMISSION = find_col(raw, ["Commission"], required=False)
COL_FEE = find_col(raw, ["Fee"], required=False)
COL_SWAP = find_col(raw, ["Swap"], required=False)
COL_PROFIT = find_col(raw, ["Profit", "P/L", "PnL", "P&L"])

deals = raw.copy()

deals["time"] = pd.to_datetime(deals[COL_TIME], errors="coerce", format="%Y.%m.%d %H:%M:%S")
if deals["time"].isna().all():
    deals["time"] = pd.to_datetime(deals[COL_TIME], errors="coerce")

deals["deal_id"] = deals[COL_DEAL].map(clean_id) if COL_DEAL else deals.index.astype(str)
deals["symbol"] = deals[COL_SYMBOL].astype(str).str.strip()
deals["deal_type"] = deals[COL_TYPE].astype(str).str.strip().str.lower()
deals["direction"] = deals[COL_DIRECTION].astype(str).str.strip().str.lower()
deals["volume"] = deals[COL_VOLUME].map(clean_number)
deals["price"] = deals[COL_PRICE].map(clean_number)
deals["order_id"] = deals[COL_ORDER].map(clean_id) if COL_ORDER else deals["deal_id"]

for source_col, clean_col in [
    (COL_PROFIT, "profit"),
    (COL_COMMISSION, "commission"),
    (COL_FEE, "fee"),
    (COL_SWAP, "swap"),
]:
    if source_col:
        deals[clean_col] = deals[source_col].map(clean_number).fillna(0.0)
    else:
        deals[clean_col] = 0.0

if INCLUDE_COMMISSION_FEE_SWAP:
    deals["net_pnl"] = deals["profit"] + deals["commission"] + deals["fee"] + deals["swap"]
else:
    deals["net_pnl"] = deals["profit"]

deals = deals.sort_values(["time", "deal_id"], na_position="last").reset_index(drop=True)

closing_directions = {"out", "in/out", "out by", "out_by", "close", "closed"}
closed_fills = deals[deals["direction"].isin(closing_directions)].copy()

# Fallback: if no Direction-based closing rows are found, use non-zero P&L rows.
if closed_fills.empty:
    print("No closing rows found via Direction. Falling back to rows with non-zero net P&L.")
    closed_fills = deals[deals["net_pnl"].abs() > 1e-12].copy()

closed_fills["position_side"] = closed_fills["deal_type"].map(side_from_closing_type)
closed_fills["hour"] = closed_fills["time"].dt.hour
closed_fills["date"] = closed_fills["time"].dt.date
closed_fills["result"] = closed_fills["net_pnl"].map(signed_label)

print(f"Raw rows: {len(deals):,}")
print(f"Closed fills: {len(closed_fills):,}")
print(f"Symbols: {', '.join(sorted(deals['symbol'].dropna().unique()))}")

display(closed_fills.head())

In [ ]:
# =========================
# Aggregate multiple fills into clean closing-order events
# =========================

def aggregate_exit_events(closed: pd.DataFrame) -> pd.DataFrame:
    if closed.empty:
        return closed.copy()

    group_cols = ["order_id", "symbol", "deal_type", "direction"] if AGGREGATE_BY_CLOSING_ORDER else ["deal_id"]

    rows = []
    for group_key, g in closed.groupby(group_cols, dropna=False, sort=False):
        row = {
            "event_id": "|".join(map(str, group_key)) if isinstance(group_key, tuple) else str(group_key),
            "close_time": g["time"].max(),
            "first_fill_time": g["time"].min(),
            "symbol": g["symbol"].iloc[0],
            "closing_type": g["deal_type"].iloc[0],
            "direction": g["direction"].iloc[0],
            "position_side": g["position_side"].iloc[0],
            "fills": len(g),
            "volume": g["volume"].sum(),
            "avg_close_price": weighted_avg(g["price"], g["volume"]),
            "profit": g["profit"].sum(),
            "commission": g["commission"].sum(),
            "fee": g["fee"].sum(),
            "swap": g["swap"].sum(),
            "net_pnl": g["net_pnl"].sum(),
            "deal_ids": ", ".join(g["deal_id"].astype(str).tolist()),
            "order_ids": ", ".join(sorted(set(g["order_id"].astype(str).tolist()))),
        }
        rows.append(row)

    events = pd.DataFrame(rows)
    events = events.sort_values(["close_time", "event_id"], na_position="last").reset_index(drop=True)
    events["event_no"] = np.arange(1, len(events) + 1)
    events["hour"] = events["close_time"].dt.hour
    events["date"] = events["close_time"].dt.date
    events["result"] = events["net_pnl"].map(signed_label)
    events["cum_pnl"] = events["net_pnl"].cumsum()
    events["running_peak"] = np.maximum.accumulate(np.r_[0, events["cum_pnl"].values])[1:]
    events["drawdown"] = events["cum_pnl"] - events["running_peak"]
    events["is_win"] = events["net_pnl"] > 0
    events["is_loss"] = events["net_pnl"] < 0
    events["is_breakeven"] = events["net_pnl"] == 0
    return events

exit_events = aggregate_exit_events(closed_fills)

print(f"Closing-order events: {len(exit_events):,}")
display(exit_events.head())
display(exit_events.tail())

In [ ]:
# =========================
# Full statistics
# =========================

def equity_drawdown(pnl_series: pd.Series):
    pnl = pd.Series(pnl_series).fillna(0.0).astype(float)
    equity = pnl.cumsum()
    equity_with_start = pd.Series([0.0] + equity.tolist())
    peak = equity_with_start.cummax()
    dd = equity_with_start - peak
    return equity, peak.iloc[1:].reset_index(drop=True), dd.iloc[1:].reset_index(drop=True), dd.min()

def calculate_stats(events: pd.DataFrame, raw_deals: pd.DataFrame, fills: pd.DataFrame) -> dict:
    pnl = events["net_pnl"].astype(float) if not events.empty else pd.Series(dtype=float)
    wins = pnl[pnl > 0]
    losses = pnl[pnl < 0]
    breakeven = pnl[pnl == 0]
    equity, peak, dd, max_dd = equity_drawdown(pnl)
    signs = pnl.map(lambda x: "W" if x > 0 else ("L" if x < 0 else "BE")).tolist()

    gross_profit = wins.sum()
    gross_loss = losses.sum()
    total_net = pnl.sum()
    n = len(pnl)

    peak_idx = int(equity.idxmax()) if len(equity) else None
    trough_idx = int(dd.idxmin()) if len(dd) else None
    largest_giveback_from_peak_to_close = (equity.max() - equity.iloc[-1]) if len(equity) else np.nan

    stats = {
        "source_file": FILE_NAME,
        "start_time": raw_deals["time"].min(),
        "end_time": raw_deals["time"].max(),
        "raw_rows": len(raw_deals),
        "closed_fills": len(fills),
        "closing_order_events": n,
        "symbols_traded": raw_deals["symbol"].nunique(),
        "symbols": ", ".join(sorted(raw_deals["symbol"].dropna().unique())),
        "total_volume_closed": fills["volume"].sum() if "volume" in fills else np.nan,
        "net_pnl": total_net,
        "gross_profit": gross_profit,
        "gross_loss": gross_loss,
        "profit_factor": safe_div(gross_profit, abs(gross_loss)),
        "win_count": len(wins),
        "loss_count": len(losses),
        "breakeven_count": len(breakeven),
        "win_rate": safe_div(len(wins), n),
        "loss_rate": safe_div(len(losses), n),
        "avg_trade": pnl.mean() if n else np.nan,
        "median_trade": pnl.median() if n else np.nan,
        "avg_win": wins.mean() if len(wins) else np.nan,
        "avg_loss": losses.mean() if len(losses) else np.nan,
        "median_win": wins.median() if len(wins) else np.nan,
        "median_loss": losses.median() if len(losses) else np.nan,
        "best_trade": pnl.max() if n else np.nan,
        "worst_trade": pnl.min() if n else np.nan,
        "payoff_ratio": safe_div(wins.mean() if len(wins) else np.nan, abs(losses.mean()) if len(losses) else np.nan),
        "expectancy": pnl.mean() if n else np.nan,
        "expectancy_per_closed_fill": safe_div(total_net, len(fills)),
        "std_trade": pnl.std(ddof=1) if n > 1 else np.nan,
        "trade_sharpe_like": safe_div(pnl.mean(), pnl.std(ddof=1)) * math.sqrt(n) if n > 1 and pnl.std(ddof=1) != 0 else np.nan,
        "max_drawdown": max_dd,
        "max_drawdown_pct_initial_equity": safe_div(max_dd, INITIAL_EQUITY) if INITIAL_EQUITY else np.nan,
        "recovery_factor": safe_div(total_net, abs(max_dd)),
        "largest_giveback_peak_to_close": largest_giveback_from_peak_to_close,
        "max_win_streak": max_streak(signs, "W"),
        "max_loss_streak": max_streak(signs, "L"),
        "peak_equity": equity.max() if len(equity) else np.nan,
        "final_equity_pnl": equity.iloc[-1] if len(equity) else np.nan,
        "peak_event_no": int(events.loc[peak_idx, "event_no"]) if peak_idx is not None and len(events) else None,
        "trough_event_no": int(events.loc[trough_idx, "event_no"]) if trough_idx is not None and len(events) else None,
    }

    if INITIAL_EQUITY:
        stats["return_on_initial_equity"] = total_net / INITIAL_EQUITY
        stats["ending_equity"] = INITIAL_EQUITY + total_net
    else:
        stats["return_on_initial_equity"] = np.nan
        stats["ending_equity"] = np.nan

    return stats

summary_stats = calculate_stats(exit_events, deals, closed_fills)

def stats_to_display(stats):
    rows = [
        ("Source file", stats["source_file"]),
        ("Time span", f"{stats['start_time']} → {stats['end_time']}"),
        ("Raw rows", f"{stats['raw_rows']:,}"),
        ("Closed fills", f"{stats['closed_fills']:,}"),
        ("Closing-order events", f"{stats['closing_order_events']:,}"),
        ("Symbols traded", f"{stats['symbols_traded']:,}: {stats['symbols']}"),
        ("Closed volume", f"{stats['total_volume_closed']:,.2f}"),
        ("Net P&L", money(stats["net_pnl"])),
        ("Gross profit", money(stats["gross_profit"])),
        ("Gross loss", money(stats["gross_loss"])),
        ("Profit factor", f"{stats['profit_factor']:.2f}" if pd.notna(stats["profit_factor"]) else "n/a"),
        ("Win rate", pct(stats["win_rate"])),
        ("Wins / Losses / Breakeven", f"{stats['win_count']} / {stats['loss_count']} / {stats['breakeven_count']}"),
        ("Average trade", money(stats["avg_trade"])),
        ("Median trade", money(stats["median_trade"])),
        ("Average winner", money(stats["avg_win"])),
        ("Average loser", money(stats["avg_loss"])),
        ("Payoff ratio", f"{stats['payoff_ratio']:.2f}" if pd.notna(stats["payoff_ratio"]) else "n/a"),
        ("Expectancy per event", money(stats["expectancy"])),
        ("Expectancy per closed fill", money(stats["expectancy_per_closed_fill"])),
        ("Best trade/event", money(stats["best_trade"])),
        ("Worst trade/event", money(stats["worst_trade"])),
        ("Std. dev. per event", money(stats["std_trade"])),
        ("Trade Sharpe-like", f"{stats['trade_sharpe_like']:.2f}" if pd.notna(stats["trade_sharpe_like"]) else "n/a"),
        ("Max drawdown", money(stats["max_drawdown"])),
        ("Recovery factor", f"{stats['recovery_factor']:.2f}" if pd.notna(stats["recovery_factor"]) else "n/a"),
        ("Peak P&L", money(stats["peak_equity"])),
        ("Largest giveback from peak to close", money(stats["largest_giveback_peak_to_close"])),
        ("Max win streak", f"{stats['max_win_streak']:,}"),
        ("Max loss streak", f"{stats['max_loss_streak']:,}"),
    ]
    if INITIAL_EQUITY:
        rows += [
            ("Initial equity", money(INITIAL_EQUITY)),
            ("Ending equity", money(stats["ending_equity"])),
            ("Return on initial equity", pct(stats["return_on_initial_equity"])),
            ("Max DD % of initial equity", pct(stats["max_drawdown_pct_initial_equity"])),
        ]
    return pd.DataFrame(rows, columns=["Metric", "Value"])

summary_display = stats_to_display(summary_stats)
display(summary_display)

In [ ]:
# =========================
# Group statistics
# =========================

def group_stats(events: pd.DataFrame, group_col: str) -> pd.DataFrame:
    if events.empty:
        return pd.DataFrame()

    rows = []
    for key, g in events.groupby(group_col, dropna=False):
        pnl = g["net_pnl"].astype(float)
        wins = pnl[pnl > 0]
        losses = pnl[pnl < 0]
        gross_profit = wins.sum()
        gross_loss = losses.sum()
        rows.append({
            group_col: key,
            "events": len(g),
            "fills": g["fills"].sum() if "fills" in g else np.nan,
            "volume": g["volume"].sum() if "volume" in g else np.nan,
            "wins": len(wins),
            "losses": len(losses),
            "breakeven": (pnl == 0).sum(),
            "win_rate": safe_div(len(wins), len(g)),
            "net_pnl": pnl.sum(),
            "gross_profit": gross_profit,
            "gross_loss": gross_loss,
            "profit_factor": safe_div(gross_profit, abs(gross_loss)),
            "avg_event": pnl.mean(),
            "median_event": pnl.median(),
            "avg_win": wins.mean() if len(wins) else np.nan,
            "avg_loss": losses.mean() if len(losses) else np.nan,
            "best_event": pnl.max(),
            "worst_event": pnl.min(),
        })
    out = pd.DataFrame(rows).sort_values("net_pnl", ascending=False).reset_index(drop=True)
    return out

symbol_stats = group_stats(exit_events, "symbol")
hour_stats = group_stats(exit_events, "hour")
side_stats = group_stats(exit_events, "position_side")
type_stats = group_stats(exit_events, "closing_type")

# Add a 0-padded hour label for cleaner charts.
if not hour_stats.empty:
    hour_stats["hour_label"] = hour_stats["hour"].astype(int).map(lambda h: f"{h:02d}:00")
    hour_stats = hour_stats.sort_values("hour").reset_index(drop=True)

display(Markdown("### Symbol stats"))
display(symbol_stats)

display(Markdown("### Hour stats"))
display(hour_stats)

display(Markdown("### Position-side stats"))
display(side_stats)

display(Markdown("### Closing-type stats"))
display(type_stats)

In [ ]:
# =========================
# Best/worst trades and diagnostic tables
# =========================

best_events = exit_events.sort_values("net_pnl", ascending=False).head(TOP_N_TABLES).copy()
worst_events = exit_events.sort_values("net_pnl", ascending=True).head(TOP_N_TABLES).copy()

display(Markdown(f"### Top {TOP_N_TABLES} best closing-order events"))
display(best_events[["event_no", "close_time", "symbol", "position_side", "volume", "avg_close_price", "fills", "net_pnl", "cum_pnl", "deal_ids"]])

display(Markdown(f"### Top {TOP_N_TABLES} worst closing-order events"))
display(worst_events[["event_no", "close_time", "symbol", "position_side", "volume", "avg_close_price", "fills", "net_pnl", "cum_pnl", "deal_ids"]])

# Optional sanity check: closed fills vs aggregated events
sanity = pd.DataFrame({
    "check": [
        "Closed fills net P&L",
        "Aggregated events net P&L",
        "Difference",
        "Closed fills count",
        "Aggregated event count",
    ],
    "value": [
        closed_fills["net_pnl"].sum(),
        exit_events["net_pnl"].sum(),
        closed_fills["net_pnl"].sum() - exit_events["net_pnl"].sum(),
        len(closed_fills),
        len(exit_events),
    ]
})
display(Markdown("### Reconciliation check"))
display(sanity)

In [ ]:
# =========================
# Charts
# =========================

def savefig(name, tight=True):
    path = f"{ASSET_DIR}/{name}"
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    plt.close()
    return path

chart_paths = {}

if exit_events.empty:
    raise ValueError("No closing events available for charting.")

# 1. Equity curve
plt.figure(figsize=FIGSIZE_WIDE)
plt.plot(exit_events["event_no"], exit_events["cum_pnl"], marker="o", linewidth=1.8, markersize=3.5)
plt.axhline(0, linewidth=1)
plt.title("Equity curve: cumulative P&L by closing-order event")
plt.xlabel("Closing-order event #")
plt.ylabel(f"Cumulative P&L ({CURRENCY_SYMBOL or 'account currency'})")
plt.grid(True, alpha=0.3)
chart_paths["equity_curve"] = savefig("01_equity_curve.png")

# 2. Drawdown
plt.figure(figsize=FIGSIZE_WIDE)
plt.fill_between(exit_events["event_no"], exit_events["drawdown"], 0, alpha=0.35)
plt.plot(exit_events["event_no"], exit_events["drawdown"], linewidth=1.5)
plt.axhline(0, linewidth=1)
plt.title("Drawdown from intraday P&L peak")
plt.xlabel("Closing-order event #")
plt.ylabel("Drawdown")
plt.grid(True, alpha=0.3)
chart_paths["drawdown"] = savefig("02_drawdown.png")

# 3. P&L by symbol
sym_plot = symbol_stats.sort_values("net_pnl", ascending=True)
plt.figure(figsize=FIGSIZE_TALL)
plt.barh(sym_plot["symbol"].astype(str), sym_plot["net_pnl"])
plt.axvline(0, linewidth=1)
plt.title("Net P&L by symbol")
plt.xlabel("Net P&L")
plt.ylabel("Symbol")
plt.grid(axis="x", alpha=0.3)
chart_paths["pnl_by_symbol"] = savefig("03_pnl_by_symbol.png")

# 4. P&L by hour
hour_plot = hour_stats.sort_values("hour")
plt.figure(figsize=FIGSIZE_WIDE)
plt.bar(hour_plot["hour_label"].astype(str), hour_plot["net_pnl"])
plt.axhline(0, linewidth=1)
plt.title("Net P&L by hour")
plt.xlabel("Hour")
plt.ylabel("Net P&L")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
chart_paths["pnl_by_hour"] = savefig("04_pnl_by_hour.png")

# 5. P&L distribution
plt.figure(figsize=FIGSIZE_TALL)
plt.hist(exit_events["net_pnl"], bins=min(20, max(5, int(np.sqrt(len(exit_events))) + 3)), edgecolor="black", alpha=0.8)
plt.axvline(0, linewidth=1)
plt.axvline(exit_events["net_pnl"].mean(), linestyle="--", linewidth=1.5, label=f"Mean: {exit_events['net_pnl'].mean():.2f}")
plt.title("Distribution of closing-order event P&L")
plt.xlabel("Event P&L")
plt.ylabel("Frequency")
plt.legend()
plt.grid(axis="y", alpha=0.3)
chart_paths["pnl_distribution"] = savefig("05_pnl_distribution.png")

# 6. Wins/losses by symbol
wl = exit_events.pivot_table(index="symbol", columns="result", values="event_id", aggfunc="count", fill_value=0)
for col in ["Win", "Loss", "Breakeven"]:
    if col not in wl.columns:
        wl[col] = 0
wl = wl[["Win", "Loss", "Breakeven"]].sort_index()
plt.figure(figsize=FIGSIZE_TALL)
bottom = np.zeros(len(wl))
for col in wl.columns:
    plt.bar(wl.index.astype(str), wl[col], bottom=bottom, label=col)
    bottom += wl[col].values
plt.title("Win/loss count by symbol")
plt.xlabel("Symbol")
plt.ylabel("Number of events")
plt.legend()
plt.grid(axis="y", alpha=0.3)
chart_paths["win_loss_by_symbol"] = savefig("06_win_loss_by_symbol.png")

# 7. Trade-by-trade event P&L
plt.figure(figsize=FIGSIZE_WIDE)
plt.bar(exit_events["event_no"], exit_events["net_pnl"])
plt.axhline(0, linewidth=1)
plt.title("P&L by closing-order event")
plt.xlabel("Closing-order event #")
plt.ylabel("Event P&L")
plt.grid(axis="y", alpha=0.3)
chart_paths["event_pnl_sequence"] = savefig("07_event_pnl_sequence.png")

# 8. Position-side P&L
if not side_stats.empty:
    side_plot = side_stats.sort_values("net_pnl", ascending=True)
    plt.figure(figsize=FIGSIZE_TALL)
    plt.barh(side_plot["position_side"].astype(str), side_plot["net_pnl"])
    plt.axvline(0, linewidth=1)
    plt.title("Net P&L by inferred position side")
    plt.xlabel("Net P&L")
    plt.ylabel("Position side")
    plt.grid(axis="x", alpha=0.3)
    chart_paths["pnl_by_side"] = savefig("08_pnl_by_side.png")

# 9. Volume vs P&L
plt.figure(figsize=FIGSIZE_TALL)
plt.scatter(exit_events["volume"], exit_events["net_pnl"], alpha=0.8)
plt.axhline(0, linewidth=1)
plt.title("Volume vs event P&L")
plt.xlabel("Closed volume")
plt.ylabel("Event P&L")
plt.grid(True, alpha=0.3)
chart_paths["volume_vs_pnl"] = savefig("09_volume_vs_pnl.png")

chart_paths

In [ ]:
# =========================
# Automated written readout
# =========================

def make_readout(stats, symbol_stats, hour_stats, events):
    lines = []

    net = stats["net_pnl"]
    pf = stats["profit_factor"]
    wr = stats["win_rate"]
    max_dd = stats["max_drawdown"]
    recovery = stats["recovery_factor"]

    lines.append(f"Net result: {money(net)} across {stats['closing_order_events']} aggregated closing-order events.")
    lines.append(f"Profit factor: {pf:.2f}. Win rate: {wr:.1%}. Expectancy per event: {money(stats['expectancy'])}.")
    lines.append(f"Max intraday drawdown: {money(max_dd)}. Recovery factor: {recovery:.2f}.")

    if not symbol_stats.empty:
        best_sym = symbol_stats.iloc[0]
        worst_sym = symbol_stats.sort_values("net_pnl").iloc[0]
        lines.append(f"Best symbol: {best_sym['symbol']} with {money(best_sym['net_pnl'])}.")
        lines.append(f"Worst symbol: {worst_sym['symbol']} with {money(worst_sym['net_pnl'])}.")

    if not hour_stats.empty:
        best_hour = hour_stats.sort_values("net_pnl", ascending=False).iloc[0]
        worst_hour = hour_stats.sort_values("net_pnl", ascending=True).iloc[0]
        lines.append(f"Best hour: {int(best_hour['hour']):02d}:00 with {money(best_hour['net_pnl'])}.")
        lines.append(f"Worst hour: {int(worst_hour['hour']):02d}:00 with {money(worst_hour['net_pnl'])}.")
        net_without_best_hour = stats["net_pnl"] - best_hour["net_pnl"]
        lines.append(f"Net P&L without the best hour would have been {money(net_without_best_hour)}.")

    if len(events):
        peak_event = events.loc[events["cum_pnl"].idxmax()]
        final = events["cum_pnl"].iloc[-1]
        giveback = peak_event["cum_pnl"] - final
        lines.append(f"Peak P&L reached {money(peak_event['cum_pnl'])} at event #{int(peak_event['event_no'])}; peak-to-close giveback was {money(giveback)}.")

    # Risk flags
    flags = []
    if pd.notna(max_dd) and abs(max_dd) > 0 and abs(max_dd) > max(abs(net) * 0.5, 1e-9):
        flags.append("Drawdown was large relative to final profit; consider a daily max-giveback rule.")
    if not symbol_stats.empty:
        worst_sym = symbol_stats.sort_values("net_pnl").iloc[0]
        if worst_sym["net_pnl"] < 0 and abs(worst_sym["net_pnl"]) > max(abs(net) * 0.25, 1e-9):
            flags.append(f"{worst_sym['symbol']} created a meaningful drag; review entries, timing, and stop discipline for that symbol.")
    if stats["max_loss_streak"] >= 3:
        flags.append(f"Loss streak reached {stats['max_loss_streak']} events; consider a pause/recheck rule after 2 consecutive losses.")
    if flags:
        lines.append("Risk/process flags: " + " ".join(flags))

    return "\n".join(f"- {line}" for line in lines)

readout_md = make_readout(summary_stats, symbol_stats, hour_stats, exit_events)
display(Markdown("## Automated readout"))
display(Markdown(readout_md))

In [ ]:
# =========================
# Export CSVs and HTML report
# =========================

def format_df_for_html(df, max_rows=30):
    if df is None or df.empty:
        return "<p><em>No data.</em></p>"
    temp = df.copy()
    for col in temp.columns:
        if pd.api.types.is_float_dtype(temp[col]):
            temp[col] = temp[col].map(lambda x: "" if pd.isna(x) else f"{x:,.2f}")
        elif pd.api.types.is_datetime64_any_dtype(temp[col]):
            temp[col] = temp[col].astype(str)
    return temp.head(max_rows).to_html(index=False, classes="data-table", border=0)

def image_to_base64(path):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/png;base64,{data}"

# Write CSV outputs
closed_fills_path = f"{OUTPUT_DIR}/closed_fills_clean.csv"
events_path = f"{OUTPUT_DIR}/exit_events.csv"
symbol_stats_path = f"{OUTPUT_DIR}/symbol_stats.csv"
hour_stats_path = f"{OUTPUT_DIR}/hour_stats.csv"
side_stats_path = f"{OUTPUT_DIR}/side_stats.csv"
summary_path = f"{OUTPUT_DIR}/summary_metrics.csv"

closed_fills.to_csv(closed_fills_path, index=False)
exit_events.to_csv(events_path, index=False)
symbol_stats.to_csv(symbol_stats_path, index=False)
hour_stats.to_csv(hour_stats_path, index=False)
side_stats.to_csv(side_stats_path, index=False)

summary_export = pd.DataFrame([summary_stats])
summary_export.to_csv(summary_path, index=False)

# Build HTML
chart_sections = []
for title, key in [
    ("Equity curve", "equity_curve"),
    ("Drawdown", "drawdown"),
    ("Net P&L by symbol", "pnl_by_symbol"),
    ("Net P&L by hour", "pnl_by_hour"),
    ("P&L distribution", "pnl_distribution"),
    ("Win/loss by symbol", "win_loss_by_symbol"),
    ("Event P&L sequence", "event_pnl_sequence"),
    ("P&L by side", "pnl_by_side"),
    ("Volume vs P&L", "volume_vs_pnl"),
]:
    if key in chart_paths:
        chart_sections.append(f"""
        <section class="card">
          <h2>{title}</h2>
          <img src="{image_to_base64(chart_paths[key])}" alt="{title}">
        </section>
        """)

html = f"""
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>Day Trading Analysis Report</title>
<style>
body {{
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
  margin: 0;
  background: #f6f7f9;
  color: #202124;
}}
header {{
  background: #111827;
  color: white;
  padding: 28px 36px;
}}
main {{
  max-width: 1180px;
  margin: 0 auto;
  padding: 24px;
}}
.card {{
  background: white;
  border-radius: 14px;
  padding: 22px;
  margin: 18px 0;
  box-shadow: 0 1px 8px rgba(0,0,0,.08);
}}
.kpi-grid {{
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
  gap: 12px;
}}
.kpi {{
  background: #f3f4f6;
  border-radius: 12px;
  padding: 14px;
}}
.kpi .label {{
  color: #60646c;
  font-size: 13px;
}}
.kpi .value {{
  font-size: 24px;
  font-weight: 700;
  margin-top: 5px;
}}
img {{
  max-width: 100%;
  height: auto;
  display: block;
  margin: 8px auto;
}}
.data-table {{
  border-collapse: collapse;
  width: 100%;
  font-size: 13px;
}}
.data-table th {{
  text-align: left;
  background: #f3f4f6;
  padding: 8px;
  border-bottom: 1px solid #d8dce2;
}}
.data-table td {{
  padding: 8px;
  border-bottom: 1px solid #edf0f2;
}}
pre {{
  white-space: pre-wrap;
  background: #f3f4f6;
  border-radius: 12px;
  padding: 14px;
}}
</style>
</head>
<body>
<header>
  <h1>Day Trading Analysis Report</h1>
  <p>Source file: {FILE_NAME} · Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}</p>
</header>
<main>
  <section class="card">
    <h2>Headline KPIs</h2>
    <div class="kpi-grid">
      <div class="kpi"><div class="label">Net P&L</div><div class="value">{money(summary_stats["net_pnl"])}</div></div>
      <div class="kpi"><div class="label">Profit factor</div><div class="value">{summary_stats["profit_factor"]:.2f}</div></div>
      <div class="kpi"><div class="label">Win rate</div><div class="value">{summary_stats["win_rate"]:.1%}</div></div>
      <div class="kpi"><div class="label">Expectancy/event</div><div class="value">{money(summary_stats["expectancy"])}</div></div>
      <div class="kpi"><div class="label">Max drawdown</div><div class="value">{money(summary_stats["max_drawdown"])}</div></div>
      <div class="kpi"><div class="label">Events</div><div class="value">{summary_stats["closing_order_events"]:,}</div></div>
    </div>
  </section>

  <section class="card">
    <h2>Automated readout</h2>
    <pre>{readout_md}</pre>
  </section>

  <section class="card">
    <h2>Full summary stats</h2>
    {format_df_for_html(summary_display, max_rows=100)}
  </section>

  {''.join(chart_sections)}

  <section class="card">
    <h2>Symbol stats</h2>
    {format_df_for_html(symbol_stats, max_rows=100)}
  </section>

  <section class="card">
    <h2>Hourly stats</h2>
    {format_df_for_html(hour_stats, max_rows=100)}
  </section>

  <section class="card">
    <h2>Position-side stats</h2>
    {format_df_for_html(side_stats, max_rows=100)}
  </section>

  <section class="card">
    <h2>Top {TOP_N_TABLES} best events</h2>
    {format_df_for_html(best_events[["event_no", "close_time", "symbol", "position_side", "volume", "avg_close_price", "fills", "net_pnl", "cum_pnl", "deal_ids"]], max_rows=TOP_N_TABLES)}
  </section>

  <section class="card">
    <h2>Top {TOP_N_TABLES} worst events</h2>
    {format_df_for_html(worst_events[["event_no", "close_time", "symbol", "position_side", "volume", "avg_close_price", "fills", "net_pnl", "cum_pnl", "deal_ids"]], max_rows=TOP_N_TABLES)}
  </section>
</main>
</body>
</html>
"""

html_path = f"{OUTPUT_DIR}/day_trading_analysis_report.html"
with open(html_path, "w", encoding="utf-8") as f:
    f.write(html)

# Zip bundle
zip_path = "day_trading_analysis_bundle.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in Path(OUTPUT_DIR).rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(Path(OUTPUT_DIR).parent)))

print("Created outputs:")
for p in [html_path, zip_path, closed_fills_path, events_path, symbol_stats_path, hour_stats_path, side_stats_path, summary_path]:
    print(" -", p)

display(HTML(f'<p><a href="{html_path}" target="_blank">Open HTML report</a></p>'))

# In Colab, uncomment these lines to download automatically:
# from google.colab import files
# files.download(zip_path)

## Notes and assumptions

- The notebook assumes the CSV is a MetaTrader-style deal history export with columns such as `Time`, `Deal`, `Symbol`, `Type`, `Direction`, `Volume`, `Price`, `Order`, `Commission`, `Fee`, `Swap`, and `Profit`.
- Closing trades are identified using `Direction` values such as `out`, `in/out`, `out by`, or similar. If no such rows are found, the notebook falls back to non-zero P&L rows.
- Multiple fills under the same closing order are aggregated into one event by default. Change `AGGREGATE_BY_CLOSING_ORDER = False` to analyze each fill separately.
- `net_pnl` includes `Profit + Commission + Fee + Swap` by default. Change `INCLUDE_COMMISSION_FEE_SWAP = False` to use the raw `Profit` column only.
- Starting account equity is optional. Set `INITIAL_EQUITY` in the configuration cell if you want return % and drawdown %.
